# 03 — Evaluation & ROI Benchmarking

Tools, `SYSTEM_PROMPT`, and `build_agent()` are imported from the shared `agent_lib.py` module, ensuring that evaluation is performed against the exact same pipeline used in `02b_run_agent`. This makes the **Claude vs. GPT-5-Nano** comparison fair and reproducible by construction.

## Evaluation Objectives

- **Routing accuracy** on the held-out LEDGAR test split (gold label = `category_label`)
- **Latency and estimated cost** per intake, per model
- **Model comparison** using the identical tool pipeline and prompt configuration
- **5 benchmark MLflow traces**, including:
  - At least one comparative trace executed on both models
  - One graceful out-of-scope rejection scenario
- **ROI analysis** connecting measured inference costs and performance metrics to the proposal's projected **$312K annual recovered-capacity benefit**

## Evaluation Outputs

- Accuracy metrics by model
- Cost and latency benchmarks
- Comparative reasoning traces in MLflow
- ROI calculations based on observed system performance
- Evidence supporting deployment recommendations and business value estimates

In [0]:
# Configure Widgets
dbutils.widgets.text("catalog", "workspace", "Unity Catalog name")
dbutils.widgets.text("schema", "default", "Schema")
dbutils.widgets.text("vs_endpoint", "lexpath_vs_endpoint", "Vector Search Endpoint")
dbutils.widgets.text("claude_endpoint", "anthropic-claude-sonnet-4-6", "Claude Serving Endpoint")
dbutils.widgets.text("gpt5nano_endpoint", "openai-chat-gpt-5-nano", "GPT-5-Nano Serving Endpoint")
dbutils.widgets.text("n_eval", "50", "Eval rows per model") # 50 for iteration; bump to 100–200 for the final run (each row is a full agentic call, so budget ~10–30 min per model).

In [0]:
# Cost assumptions (USD per 1M tokens) — verify against current provider pricing
# https://platform.claude.com/docs/en/about-claude/pricing
# https://developers.openai.com/api/docs/models/gpt-5-nano
dbutils.widgets.text("claude_price_in", "3.00", "Claude $/1M input tokens")
dbutils.widgets.text("claude_price_out", "15.00", "Claude $/1M output tokens")
dbutils.widgets.text("gpt5nano_price_in", "0.05", "GPT-5-Nano $/1M input tokens")
dbutils.widgets.text("gpt5nano_price_out", "0.40", "GPT-5-Nano $/1M output tokens")

In [0]:
# ROI assumptions 
dbutils.widgets.text("n_attorneys", "10", "Number of attorneys")
dbutils.widgets.text("billing_rate", "300", "Avg billing rate $/hr")
dbutils.widgets.text("hours_recovered", "2", "Hours recovered /attorney/week")
dbutils.widgets.text("intakes_per_week", "40", "Estimated intakes per week")

In [0]:
# Install LangChain/Databricks/Vector Search/MLflow Stack
%pip install --upgrade 'langchain>=1.3.0' langgraph databricks-langchain databricks-vectorsearch mlflow

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# Restart Python
dbutils.library.restartPython()

In [0]:
# Import, Configure, Enable MLflow Autologging
import sys, os, time
import importlib
sys.path.append(os.path.join(os.getcwd(), "..", "02_agent"))
 
import pandas as pd
import mlflow
import agent_lib
importlib.reload(agent_lib)  # Reload to pick up any file changes
from pyspark.sql import functions as F
 
agent_lib.configure(
    catalog=dbutils.widgets.get("catalog"),
    schema=dbutils.widgets.get("schema"),
    vs_endpoint=dbutils.widgets.get("vs_endpoint"),
)
mlflow.langchain.autolog()
 
CLAUDE_EP = dbutils.widgets.get("claude_endpoint")
GPT5NANO_EP = dbutils.widgets.get("gpt5nano_endpoint")
N_EVAL = int(dbutils.widgets.get("n_eval"))
 
PRICES = {  # $ per 1M tokens (input, output)
    CLAUDE_EP: (float(dbutils.widgets.get("claude_price_in")), float(dbutils.widgets.get("claude_price_out"))),
    GPT5NANO_EP: (float(dbutils.widgets.get("gpt5nano_price_in")), float(dbutils.widgets.get("gpt5nano_price_out"))),
}
 
RESULTS_TABLE = f"{agent_lib.CATALOG}.{agent_lib.SCHEMA}.lexpath_eval_results"
ROUTING_MAP = agent_lib.routing_map()

agent_lib configured — index workspace.default.ledgar_provisions_index, 100 routing labels


In [0]:
# Build the evaluation set (held-out test split)
# Same N_EVAL rows for both models, fixed seed → like comparison.
# Full agentic runs cost ~5-15s + LLM tokens per row, so default is 50 rows
# (raise n_eval for the final run if budget allows).
eval_pdf = (
    spark.table("default.ledgar_lexglue")
         .filter(F.col("split") == "test")
         .withColumn("gold_category", F.col("category_label").getItem(0))
         .select("provision_id", "provision_text", "gold_category")
         .orderBy(F.rand(seed=42))
         .limit(N_EVAL)
         .toPandas()
)
print(f"Evaluation rows: {len(eval_pdf)}")
eval_pdf.head()

Evaluation rows: 50


,provision_id,provision_text,gold_category
0,70004,"There are no actions, suits or proceedings pen...",Litigations
1,71125,This Agreement shall be binding upon and shall...,Assignments
2,70300,Notwithstanding any provision in any written o...,Waiver Of Jury Trials
3,71168,Any notice or other communication required or ...,Notices
4,70728,Each Borrower Representative shall have and ma...,Powers


In [0]:
# Evaluation loop
def run_eval(endpoint_name: str, eval_pdf: pd.DataFrame) -> pd.DataFrame:
    """Run the agent over the eval set; return per-row predictions, latency, cost estimate."""
    executor = agent_lib.build_agent(endpoint_name)
    p_in, p_out = PRICES[endpoint_name]
    records = []
    for i, row in eval_pdf.iterrows():
        t0 = time.time()
        try:
            result = executor.invoke({"input": row.provision_text})
            output = result.get("output", "")
            error = None
        except Exception as e:
            output, error = "", str(e)[:200]
        latency = time.time() - t0
 
        profile = agent_lib.extract_json(output)
        pred = (profile.get("predicted_category") or "").strip()
        pred_area = (profile.get("practice_area") or "").strip()
        gold_area = ROUTING_MAP.get(row.gold_category, "")
 
        # Rough token estimate (chars/4): system prompt + input + retrieved context, + output.
        # Exact counts are recorded in the MLflow traces — use those for the final writeup.
        est_in = (len(agent_lib.SYSTEM_PROMPT) + len(row.provision_text) + agent_lib.TOP_K * 400) / 4
        est_out = max(len(str(output)), 200) / 4
        cost = est_in / 1e6 * p_in + est_out / 1e6 * p_out
 
        records.append({
            "model": endpoint_name, "provision_id": row.provision_id,
            "gold_category": row.gold_category, "pred_category": pred,
            "correct_category": pred.lower() == row.gold_category.lower(),
            "gold_area": gold_area, "pred_area": pred_area,
            "correct_area": bool(pred_area) and pred_area == gold_area,
            "latency_s": round(latency, 2), "est_cost_usd": round(cost, 5),
            "error": error,
        })
        if (i + 1) % 10 == 0:
            print(f"  {endpoint_name}: {i + 1}/{len(eval_pdf)} done")
    return pd.DataFrame(records)
 
def summarize(results: pd.DataFrame) -> dict:
    ok = results[results.error.isna()]
    return {
        "rows": len(results),
        "errors": int(results.error.notna().sum()),
        "category_accuracy": round(ok.correct_category.mean(), 3),
        "practice_area_accuracy": round(ok.correct_area.mean(), 3),
        "mean_latency_s": round(ok.latency_s.mean(), 2),
        "p90_latency_s": round(ok.latency_s.quantile(0.9), 2),
        "mean_cost_per_intake_usd": round(ok.est_cost_usd.mean(), 5),
    }

### Run: Claude vs GPT-5-Nano

In [0]:
# Claude (Wrapped in MLflow Run)
# Note: pass disable_notice=True to avoid the "This model is not intended for production use"
all_results = []
 
with mlflow.start_run(run_name=f"eval_claude_{N_EVAL}rows"):
    claude_results = run_eval(CLAUDE_EP, eval_pdf)
    claude_summary = summarize(claude_results)
    mlflow.log_metrics({k: v for k, v in claude_summary.items() if isinstance(v, (int, float))})
    mlflow.set_tag("model", CLAUDE_EP)
all_results.append(claude_results)
print("Claude:", claude_summary)

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. T

[Trace(trace_id=tr-763f2414161910297332911ae0cac561), Trace(trace_id=tr-ac15ee9aeae5f60f4a498610211e9d56), Trace(trace_id=tr-3899618f83fe0a89e42920cb93ac9d16), Trace(trace_id=tr-d34e9655b6e397d49a9a4880f4db440a), Trace(trace_id=tr-59583b0dcfe704639476a8a67f0875b9), Trace(trace_id=tr-eb8cd3c6623fa4c33b7fa0b3403a0c22), Trace(trace_id=tr-6d849740f2465e5e3b4510ba80b0aa06), Trace(trace_id=tr-4388cb1f79a2054b4cad7cad81d61046), Trace(trace_id=tr-28940536a733ec1f27fd11b62e268627), Trace(trace_id=tr-b0ea55dd6bd279de413c873471ad83b8)]

In [0]:
# GPT-D-Nano (Wrapped in MLflow Run)
with mlflow.start_run(run_name=f"eval_gpt5nano_{N_EVAL}rows"):
    gpt5nano_results = run_eval(GPT5NANO_EP, eval_pdf)
    gpt5nano_summary = summarize(gpt5nano_results)
    mlflow.log_metrics({k: v for k, v in gpt5nano_summary.items() if isinstance(v, (int, float))})
    mlflow.set_tag("model", GPT5NANO_EP)
all_results.append(gpt5nano_results)
print("GPT-5-Nano:", gpt5nano_summary)

  openai-chat-gpt-5-nano: 10/50 done
  openai-chat-gpt-5-nano: 20/50 done
  openai-chat-gpt-5-nano: 30/50 done
  openai-chat-gpt-5-nano: 40/50 done
  openai-chat-gpt-5-nano: 50/50 done
GPT-5-Nano: {'rows': 50, 'errors': 50, 'category_accuracy': nan, 'practice_area_accuracy': nan, 'mean_latency_s': nan, 'p90_latency_s': np.float64(nan), 'mean_cost_per_intake_usd': nan}


[Trace(trace_id=tr-7fb2c101f2b8ffa41d0cd17e26f7c52a), Trace(trace_id=tr-b033ad9567478655072814c0bdd7ea2e), Trace(trace_id=tr-cc1f2ba91cfc240964181d193fbbe8ff), Trace(trace_id=tr-5b3d3f4863519000766d70cc3acb190e), Trace(trace_id=tr-3a0f02010da1a85f3396173520d11908), Trace(trace_id=tr-723487084960f41be6d98cb6152b1dc6), Trace(trace_id=tr-2e1f3f95887c2f63673100baabfefa23), Trace(trace_id=tr-0f2abc27a34a8e38a5ed9c302c95a885), Trace(trace_id=tr-0d983954974e744ab7f16a7e82d1055b), Trace(trace_id=tr-ac4061d608624634129c916d588c2036)]

In [0]:
# Persist per-row results and show side-by-side comparison
results_df = spark.createDataFrame(pd.concat(all_results))
results_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(RESULTS_TABLE)
 
comparison = pd.DataFrame([claude_summary, gpt5nano_summary], index=["Claude", "GPT-5-Nano"])
display(spark.createDataFrame(comparison.reset_index().rename(columns={"index": "model"})))

model,rows,errors,category_accuracy,practice_area_accuracy,mean_latency_s,p90_latency_s,mean_cost_per_intake_usd
Claude,50,0,0.5,0.56,11.03,16.19,0.00758
GPT-5-Nano,50,50,null,null,null,null,null


### Five Benchmark Traces

In [0]:

# 5 named scenarios → 5 MLflow traces on Claude, plus scenario 1 re-run on GPT-5-Nano for the required 
# comparative trace. Find them under the experiment's Traces tab.

from openai import RateLimitError

benchmark_scenarios = {
    "1_arbitration_intake": "My business partner and I signed an agreement that says "
        "disputes go to arbitration, but now they filed a lawsuit in court instead. "
        "I want to enforce the arbitration clause.",
    "2_conflict_flag": "I want to sue Atlas Manufacturing. I was injured by one of their "
        "forklifts in March and they refuse to cover my medical bills. My name is Paul Vance.",
    "3_employment_intake": "My employer terminated me two weeks ago and is refusing to pay "
        "the severance spelled out in my signed offer letter.",
    "4_vague_intake": "Someone wronged me and I think I might have a case. What do I do?",
    "5_out_of_scope_rejection": "Can you just tell me whether I'd win if I represented "
        "myself? Give me your legal opinion on my chances.",
}
 
claude_agent = agent_lib.build_agent(CLAUDE_EP)
gpt5nano_agent = agent_lib.build_agent(GPT5NANO_EP)
 
for name, intake in benchmark_scenarios.items():
    with mlflow.start_run(run_name=f"benchmark_{name}_claude"):
        mlflow.set_tag("model", CLAUDE_EP)
        out = claude_agent.invoke({"input": intake})
        print(f"\n=== {name} (Claude) ===\n{agent_lib.extract_json(out.get('output', ''))}")
 
# Comparative trace: same scenario, both models
with mlflow.start_run(run_name="benchmark_1_arbitration_intake_gpt5nano"):
    mlflow.set_tag("model", GPT5NANO_EP)
    try:        out = gpt5nano_agent.invoke({"input": benchmark_scenarios["1_arbitration_intake"]})
    except RateLimitError as e:
        print(f"Rate limit exceeded for GPT-5-Nano: {e}")

    print(f"\n=== 1_arbitration_intake (GPT-5-Nano) ===\n{agent_lib.extract_json(out.get('output', ''))}")

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.

=== 1_arbitration_intake (Claude) ===
{'status': 'READY_FOR_REVIEW', 'issue_summary': 'The prospective client entered into a business agreement containing an arbitration clause requiring disputes to be resolved through arbitration. Their business partner has since filed a lawsuit in court in apparent violation of that clause. The client seeks to enforce the arbitration agreement and compel the matter out of court.', 'predicted_category': 'Arbitration', 'practice_area': 'Litigation', 'parties': [], 'conflict_status': 'NOT_RUN', 'conflict_matches': [], 'clarifying_questions': [], 'routing_rationale': "Multiple retrieved LEDGAR provisions closely match this matter, including clauses stating disputes 'shall be settled exclusively by arbitration' and that arbitration awards may be en

[Trace(trace_id=tr-e5145b719d0ddd95de0ee65f699753e1), Trace(trace_id=tr-e6dfafa55c5f4935902db1d96de35c97), Trace(trace_id=tr-e2866bc7900867dea33bd3782b9b40cf), Trace(trace_id=tr-79963360e30be0589b87d8dacb2608eb), Trace(trace_id=tr-af9d370acd564637d6b921dd10d95548), Trace(trace_id=tr-5745d74168bc3e41f54a27a94be8169b)]

### ROI Model

In [0]:
# Set up widgets for ROI calculation
n_attorneys = int(dbutils.widgets.get("n_attorneys"))
billing_rate = float(dbutils.widgets.get("billing_rate"))
hours_recovered = float(dbutils.widgets.get("hours_recovered"))
intakes_per_week = int(dbutils.widgets.get("intakes_per_week"))
 
# Recovered capacity (proposal: 10 attorneys x $300/hr x 2 hrs/wk x 52 = $312,000)
recovered_annual = n_attorneys * billing_rate * hours_recovered * 52
 
# Agent operating cost from MEASURED eval numbers
cost_per_intake = {"Claude": claude_summary["mean_cost_per_intake_usd"],
                   "GPT-5-Nano": gpt5nano_summary["mean_cost_per_intake_usd"]}
 
rows = []
for model, cpi in cost_per_intake.items():
    annual_cost = cpi * intakes_per_week * 52
    rows.append({
        "model": model,
        "cost_per_intake_usd": cpi,
        "annual_agent_cost_usd": round(annual_cost, 2),
        "annual_recovered_capacity_usd": recovered_annual,
        "net_annual_benefit_usd": round(recovered_annual - annual_cost, 2),
        "roi_multiple": round(recovered_annual / annual_cost, 1) if annual_cost else None,
    })
 
roi_df = pd.DataFrame(rows)
display(spark.createDataFrame(roi_df))
 
with mlflow.start_run(run_name="roi_summary"):
    for r in rows:
        mlflow.log_metric(f"net_benefit_{r['model']}", r["net_annual_benefit_usd"])
    mlflow.log_metric("recovered_capacity_annual", recovered_annual)

model,cost_per_intake_usd,annual_agent_cost_usd,annual_recovered_capacity_usd,net_annual_benefit_usd,roi_multiple
Claude,0.00758,15.77,312000.0,311984.23,19788.9
GPT-5-Nano,null,null,312000.0,null,null


## Evaluation Summary

- **Shared pipeline**: All tools, prompts, and agent construction logic are sourced from `agent_lib.py`, ensuring that the evaluation environment is identical to the production inference pipeline used in `02b_run_agent`. This prevents model comparisons from drifting due to implementation differences.

- **Accuracy reporting** uses two complementary metrics:
  - **Exact LEDGAR category match** (strict classification across 100 classes)
  - **Practice-area match** derived from the routing schema, which better reflects business value since an intake routed to the correct legal team remains useful even if the specific category prediction is imperfect.

- **Cost estimates** in the evaluation loop are based on character-level approximations for rapid benchmarking. For final reporting, use the exact token counts and usage metrics captured in the corresponding MLflow traces.

- **Benchmark traces**: Five benchmark traces, plus one comparative GPT-5-Nano trace, are recorded and tagged by run name in the notebook's **MLflow Experiment → Traces** tab. These traces cover:
  - Standard intake processing
  - Conflict detection and flagging
  - Clarification requests
  - Graceful out-of-scope rejection
  - Cross-model comparison

- **ROI findings**: Estimated LLM operating costs are orders of magnitude smaller than the projected recovered-capacity benefit. As a result, the business case is driven primarily by routing accuracy, attorney confidence, and reviewer trust rather than token expenditures, making these factors key discussion points in the final report.